<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №7: Адаптивная фильтрация и частотно-временной анализ

## Подготовка окружения

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import hann, gaussian, triang, boxcar
from scipy.io import wavfile
from IPython.display import Audio, display

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

In [ ]:
# Загружаем аудиофайлы
!wget -q -O fragment.wav "https://github.com/korzhimanov/dsp-seminars/raw/refs/heads/main/data/fragment.wav"
!wget -q -O sample-3s.wav "https://github.com/korzhimanov/dsp-seminars/raw/refs/heads/main/data/sample-3s.wav"

## Часть 1. Адаптивные фильтры

### Задание 1.1. Алгоритм LMS: идентификация неизвестной системы

#### Цель
Реализовать алгоритм LMS и с его помощью восстановить импульсную характеристику неизвестной системы.

#### Задание
1. «Неизвестная» система – КИХ-фильтр `h = signal.firwin(16, 0.3)`. Входной сигнал `x` – белый шум длиной 3000 отсчётов (`np.random.randn`). Желаемый сигнал – выход системы с небольшим шумом измерений: `d = signal.lfilter(h, 1, x) + 0.01*np.random.randn(len(x))`.
2. Допишите функцию `lms(x, d, N, mu)`: на каждом шаге вычисляется выход $y[n] = \mathbf{w}^T\mathbf{x}[n]$, ошибка $e[n] = d[n] - y[n]$ и обновляются коэффициенты $\mathbf{w} \leftarrow \mathbf{w} + \mu\,e[n]\,\mathbf{x}[n]$.
3. Запустите LMS с `N = 16`, `mu = 0.01`. Постройте **кривую обучения** – $10\lg e^2[n]$, сглаженную скользящим средним по 50 отсчётам.
4. Сравните найденные коэффициенты `w` с `h` на одном графике (`plt.stem`).

In [ ]:
def lms(x, d, N, mu):
    """LMS-адаптация КИХ-фильтра длины N. Возвращает коэффициенты w и сигнал ошибки e."""
    w = np.zeros(N)
    e = np.zeros(len(x))
    for n in range(N - 1, len(x)):
        x_vec = x[n - N + 1:n + 1][::-1]      # [x[n], x[n-1], ..., x[n-N+1]]
        # Ваш код здесь: выход фильтра, ошибка e[n], обновление w
        pass
    return w, e

def learning_curve(e, L=50):
    """Сглаженная кривая обучения в дБ."""
    return 10*np.log10(np.convolve(e**2, np.ones(L)/L, mode='same') + 1e-20)

np.random.seed(0)
h = signal.firwin(16, 0.3)
# Ваш код здесь

**Вопросы:**
- Через сколько итераций ошибка выходит на установившийся уровень? Чем определяется этот уровень?
- Насколько точно восстановлены коэффициенты неизвестной системы?

{Введите ваш ответ здесь}

### Задание 1.2. Выбор шага адаптации

1. Повторите задание 1.1 для `mu = 0.001, 0.01, 0.05, 0.1, 0.12, 0.15`. Постройте кривые обучения на одном графике.
2. Найдите экспериментально наибольшее значение `mu`, при котором алгоритм ещё сходится. Сравните его с оценкой $\mu_{\max} \approx \dfrac{2}{N\sigma_x^2}$, где $\sigma_x^2$ – дисперсия входного сигнала.
3. Увеличьте дисперсию входного сигнала в 4 раза (умножьте `x` на 2) и снова найдите порог устойчивости.

In [ ]:
# Ваш код здесь

**Вопросы:**
- Как шаг адаптации влияет на скорость сходимости и на установившуюся ошибку?
- Совпал ли найденный порог устойчивости с оценкой? Как он изменился при увеличении мощности входного сигнала?

{Введите ваш ответ здесь}

### Задание 1.3. NLMS при меняющейся мощности входного сигнала

1. Сформируйте вход с медленно меняющейся мощностью: `x = 3*np.random.randn(3000)*(1 + 0.9*np.sin(2*np.pi*np.arange(3000)/500))` и желаемый сигнал `d = signal.lfilter(h, 1, x)`.
2. Подберите наибольший шаг `mu`, при котором обычный LMS ещё устойчив.
3. Реализуйте NLMS: шаг нормируется на энергию вектора входных отсчётов, $\mathbf{w} \leftarrow \mathbf{w} + \dfrac{\mu}{\|\mathbf{x}[n]\|^2 + \varepsilon}\,e[n]\,\mathbf{x}[n]$, с `mu = 0.5`, `eps = 1e-6`.
4. Сравните кривые обучения LMS и NLMS.

In [ ]:
def nlms(x, d, N, mu, eps=1e-6):
    """Нормированный LMS."""
    # Ваш код здесь
    pass

# Ваш код здесь

**Вопросы:**
- Почему для LMS приходится выбирать шаг по самому громкому участку сигнала?
- Почему NLMS сходится быстрее при той же устойчивости? Какой диапазон значений `mu` допустим для NLMS?

{Введите ваш ответ здесь}

### Задание 1.4. Адаптивное подавление шума

#### Цель
Воспроизвести схему активного шумоподавления: из сигнала микрофона вычитается оценка шума, полученная из опорного датчика.

#### Задание
1. Загрузите `fragment.wav`, переведите в моно (среднее каналов) и нормируйте в диапазон $[-1, 1]$. Это полезный сигнал `s`.
2. Опорный датчик регистрирует шум `ref = 0.5*np.random.randn(len(s))`. До микрофона этот шум доходит через неизвестный акустический тракт `path = signal.firwin(32, 0.2)`. Сигнал микрофона: `d = s + signal.lfilter(path, 1, ref)`.
3. Примените LMS (вход – `ref`, желаемый сигнал – `d`, `N = 32`, `mu = 0.002`). Оценкой полезного сигнала служит **сигнал ошибки** `e`.
4. Вычислите отношение сигнал/шум до и после обработки по формуле $\mathrm{SNR} = 10\lg\dfrac{\sum s^2}{\sum (\hat s - s)^2}$ (после обработки – по второй половине записи, когда фильтр уже сошёлся).
5. Прослушайте `d` и `e`. Повторите эксперимент с NLMS и с длиной фильтра `N = 8`.

In [ ]:
fs, s = wavfile.read('fragment.wav')
s = s.astype(float).mean(axis=1)
s /= np.max(np.abs(s))

np.random.seed(1)
# Ваш код здесь

**Вопросы:**
- Почему сигнал ошибки адаптивного фильтра оказывается очищенной речью?
- Что произойдёт, если опорный датчик будет улавливать и полезный сигнал?
- Как результат зависит от соотношения длины фильтра и длины импульсной характеристики акустического тракта?

{Введите ваш ответ здесь}

### Задание 1.5. Фильтр Калмана и экспоненциальное сглаживание

1. Смоделируйте случайное блуждание `true = np.cumsum(0.1*np.random.randn(500))` и его зашумлённые измерения `z = true + 0.5*np.random.randn(500)`.
2. Реализуйте скалярный фильтр Калмана с $F = H = 1$:
   - прогноз: $P^- = P + Q$;
   - коррекция: $K = \dfrac{P^-}{P^- + R}$, $\;\hat x \leftarrow \hat x + K(z - \hat x)$, $\;P = (1 - K)P^-$.

   Используйте $Q = 0.01$, $R = 0.25$, начальные $\hat x = 0$, $P = 1$. Постройте оценку и коэффициент $K$ как функцию номера шага.
3. Найдите установившееся значение $K_\infty$ и сравните его с аналитическим $K_\infty = \dfrac{-Q + \sqrt{Q^2 + 4QR}}{2R}$.
4. Примените к измерениям экспоненциальное сглаживание с $\alpha = K_\infty$ (лекция 5) и сравните с оценкой Калмана. Вычислите среднеквадратичные ошибки обеих оценок и самих измерений.
5. Повторите для $Q = 0.1$ и $Q = 0.001$ при том же $R$.

In [ ]:
np.random.seed(2)
# Ваш код здесь

**Вопросы:**
- Почему в установившемся режиме фильтр Калмана совпадает с экспоненциальным сглаживанием? В чём тогда преимущество фильтра Калмана?
- Как отношение $Q/R$ влияет на коэффициент $K$ и на гладкость оценки?

{Введите ваш ответ здесь}

## Часть 2. Частотно-временной анализ

### Задание 2.1. Спектрограмма и выбор длины окна

1. Сформируйте сигнал длительностью 2 с при $f_s = 1000$ Гц:
   - сумма двух тонов 100 и 110 Гц на интервале $0 \le t < 1$ с;
   - два коротких щелчка (одиночные отсчёты амплитуды 5) в моменты 1.50 и 1.53 с;
   - линейный чирп от 200 до 400 Гц на всей длительности (`signal.chirp`).
2. Постройте спектрограммы с окном Ханна длиной `Nw = 32, 128, 512` отсчётов и шагом `Nw//8` (класс `ShortTimeFFT`, см. заметки).
3. Для каждой длины окна определите, различимы ли тоны 100 и 110 Гц и различимы ли два щелчка.
4. Объясните результат с помощью оценок разрешения: по частоте $\Delta f \approx 2f_s/N_w$ (ширина главного лепестка окна Ханна), по времени $\Delta t \approx N_w/f_s$.

In [ ]:
fs = 1000
t = np.arange(0, 2, 1/fs)
# Ваш код здесь: формирование сигнала

def show_spectrogram(x, fs, Nw, ax, fmax=None):
    SFT = ShortTimeFFT(hann(Nw, sym=False), hop=max(1, Nw//8), fs=fs, mfft=max(Nw, 1024))
    S = SFT.spectrogram(x)
    ax.imshow(10*np.log10(S + 1e-12), origin='lower', aspect='auto', cmap='magma',
              extent=SFT.extent(len(x)), vmin=10*np.log10(S.max()) - 60)
    ax.set_xlabel('Время, с'); ax.set_ylabel('Частота, Гц')
    if fmax:
        ax.set_ylim(0, fmax)
    ax.set_title(f'Окно {Nw} отсчётов')

# Ваш код здесь: три спектрограммы

**Вопросы:**
- При какой длине окна разрешаются тоны 100 и 110 Гц? При какой – два щелчка? Можно ли подобрать окно, при котором разрешается и то, и другое?
- Как выглядит чирп на спектрограммах с разной длиной окна и почему?

{Введите ваш ответ здесь}

### Задание 2.2. Принцип неопределённости

Эффективные длительность и ширина спектра окна $w[n]$ определяются через вторые моменты распределений $|w[n]|^2$ и $|W(f)|^2$:
$$
\Delta t^2 = \frac{\sum_n (n - \bar n)^2 |w[n]|^2}{\sum_n |w[n]|^2}, \qquad
\Delta f^2 = \frac{\int f^2 |W(f)|^2 df}{\int |W(f)|^2 df}.
$$
1. Допишите функцию `widths(w)`, вычисляющую $\Delta t$ (в отсчётах) и $\Delta f$ (в циклах на отсчёт). Спектр вычисляйте через БПФ с большим дополнением нулями (`nfft = 2**18`).
2. Вычислите произведение $\Delta t\,\Delta f$ для гауссовых окон длины 256 с σ = 8, 16, 32, а также для окон Ханна, треугольного и прямоугольного той же длины.
3. Сравните результаты с нижней границей $1/(4\pi) \approx 0.0796$.

In [ ]:
def widths(w, nfft=2**18):
    """Эффективные длительность (в отсчётах) и ширина спектра (в циклах на отсчёт) окна w."""
    # Ваш код здесь
    pass

M = 256
windows = {
    'Гаусс σ=8': gaussian(M, 8), 'Гаусс σ=16': gaussian(M, 16), 'Гаусс σ=32': gaussian(M, 32),
    'Ханн': hann(M), 'Треугольное': triang(M), 'Прямоугольное': boxcar(M),
}
# Ваш код здесь: таблица Δt, Δf, Δt·Δf

**Вопросы:**
- Какое окно ближе всего к нижней границе? Почему для гауссова окна с большим σ произведение отклоняется от границы?
- Почему у прямоугольного окна произведение получается большим и зависит от длины БПФ?

{Введите ваш ответ здесь}

### Задание 2.3. Спектрограмма музыкального фрагмента

1. Загрузите `sample-3s.wav`, переведите в моно и нормируйте.
2. Постройте осциллограмму и спектрограммы с окнами длиной 256 и 4096 отсчётов в диапазоне частот 0–4 кГц.
3. По спектрограмме с длинным окном оцените частоту основного тона ноты, звучащей около $t = 2.3$ с: найдите частоты нескольких гармоник и расстояние между ними. Уточните результат по амплитудному спектру участка длиной 8192 отсчёта (с окном Ханна).
4. Определите, какой ноте соответствует найденная частота (ля первой октавы – 440 Гц, соседние полутона отличаются в $2^{1/12}$ раз).

In [ ]:
fs_a, audio = wavfile.read('sample-3s.wav')
# Ваш код здесь

**Вопросы:**
- Что лучше видно на спектрограмме с коротким окном, а что – с длинным?
- Какова частота основного тона и какая это нота?

{Введите ваш ответ здесь}

### Задание 2.4. Обработка в частотно-временной области

1. Добавьте к музыкальному фрагменту из задания 2.3 помеху – тон 3 кГц амплитудой 0.2, звучащий только на интервале от 1 до 2 с.
2. Вычислите STFT (`SFT.stft`) с окном Ханна длиной 1024 и шагом 256. Убедитесь, что обратное преобразование (`SFT.istft(X, k1=len(x))`) без изменений восстанавливает сигнал с точностью до ошибок округления.
3. Обнулите коэффициенты STFT в полосе 3000 ± 50 Гц на интервале 0.9–2.1 с и выполните обратное преобразование.
4. Сравните спектрограммы и прослушайте сигналы до и после обработки. Вычислите отношение сигнал/помеха до и после.
5. Попробуйте вместо этого убрать помеху обычным режекторным фильтром (`signal.iirnotch`). Чем отличается результат?

In [ ]:
# Ваш код здесь

**Вопросы:**
- Почему маску удалось ограничить и по частоте, и по времени? Что произойдёт с музыкой, если обнулить ту же полосу на всей длительности?
- Чем обработка в частотно-временной области лучше обычного режекторного фильтра для помехи, которая появляется на короткое время?

{Введите ваш ответ здесь}